Build Data

In [112]:
from core.data import build_global_data

global_data = build_global_data()

Build Static Master Graph

In [113]:
from core.graph import build_master_graph

G_master = build_master_graph(global_data)

Setup Player Team Masteries (Runs once at start of game)

In [114]:
from core.hero_mastery import create_empty_masteries, set_mastery
t1_masteries = create_empty_masteries(G_master)

# Set the masteries of the heroes you have to 1
# Top 
set_mastery(t1_masteries, "Frank", "Top", 2)
set_mastery(t1_masteries, "Hakuna", "Top", 1)
set_mastery(t1_masteries, "Justice", "Top", 1)
set_mastery(t1_masteries, "Tiger Boy", "Top", 1)

# Jungle
set_mastery(t1_masteries, "Kamaitachi", "Jungler", 2)
set_mastery(t1_masteries, "Hakuna", "Jungler", 1)
set_mastery(t1_masteries, "Zealot", "Jungler", 1)
set_mastery(t1_masteries, "Xiangxi Ke", "Jungler", 1)

# Mid
set_mastery(t1_masteries, "Aurelio", "Mid", 1)
set_mastery(t1_masteries, "Xiangxi Ke", "Mid", 1)
set_mastery(t1_masteries, "Fatty White", "Mid", 1)
set_mastery(t1_masteries, "Elemi", "Mid", 1)
set_mastery(t1_masteries, "Wolfgang", "Mid", 1)

# Bot
set_mastery(t1_masteries, "Gang", "Bot", 2)
set_mastery(t1_masteries, "Niels", "Bot", 1)
set_mastery(t1_masteries, "Bariel", "Bot", 1)
set_mastery(t1_masteries, "Omaha", "Bot", 1)
set_mastery(t1_masteries, "Shougong Lei", "Bot", 1)
set_mastery(t1_masteries, "Elemi", "Bot", 1)

# Support
set_mastery(t1_masteries, "Paisai", "Support", 1)
set_mastery(t1_masteries, "Palulu", "Support", 1)
set_mastery(t1_masteries, "Dylan", "Support", 1)
set_mastery(t1_masteries, "Fatty White", "Support", 1)
set_mastery(t1_masteries, "Peiniang Zhu", "Support", 2)
set_mastery(t1_masteries, "Tiger Boy", "Support", 1)

Start of Draft (Runs before each match)

Setup Availabiltiies and Opponent Signitures

In [115]:
from core.draft import build_position_availability
from core.hero_mastery import set_mastery

# We know what we can pick, because we know our masteries
t1_available = build_position_availability(t1_masteries)

# Build out masteries to the degree you want to
t2_masteries = create_empty_masteries(G_master)

# Top
set_mastery(t2_masteries, "Miki", "Top", 1)
set_mastery(t2_masteries, "Aurelio", "Top", 1)
set_mastery(t2_masteries, "Wolfgang", "Top", 1)

# Jungler
set_mastery(t2_masteries, "Kamaitachi", "Jungler", 1)
set_mastery(t2_masteries, "Aurelio", "Jungler", 1)
set_mastery(t2_masteries, "Hakuna", "Jungler", 1)

# Mid
set_mastery(t2_masteries, "Miki", "Mid", 1)
set_mastery(t2_masteries, "Dylan", "Mid", 1)
set_mastery(t2_masteries, "Elemi", "Mid", 1)

# Bot
set_mastery(t2_masteries, "Deep Space", "Bot", 1)
set_mastery(t2_masteries, "Elemi", "Bot", 1)
set_mastery(t2_masteries, "Niels", "Bot", 1)

# Support
set_mastery(t2_masteries, "Peiniang Zhu", "Support", 1)
set_mastery(t2_masteries, "Fatty White", "Support", 1)
set_mastery(t2_masteries, "Dylan", "Support", 1)

# One the user presses confirm (either before the draft, or after they've finished in the draft, we confirm)
t2_available = build_position_availability(t2_masteries)

Apply Mastery and Signitures to Graph

In [116]:
from core.graph import confirm_hero_masteries, _clear_hero_masteries

_clear_hero_masteries(G_master)
confirm_hero_masteries(G_master, t1_masteries, t2_masteries)

Begin Draft - Recommendations

In [117]:
from core.draft import build_draft_state
draft_state = build_draft_state()
draft_state.t1_available = t1_available
draft_state.t1_picked = {}
draft_state.t2_available = t2_available
draft_state.t2_picked = {}
draft_state.banned = set()

Recommend Pick

In [ ]:
from core.draft import recommend_pick

# Recommend a Pick
best, score, explanation, flag, all_results = recommend_pick(
    G_master, 
    "t1",
    "Jungler", 
    draft_state,
    global_data
)

# Pass outputs here

print(f"Recommended: {best} ({score})")
for reason in explanation:
    output = f"- {reason}:"
    for hero in explanation[reason]:
        output += f" {hero},"
    print(output)
if flag:
    print(f"WARNING: {flag}")
print("")


Recommended: Hakuna (9)
- position_tier: A,
- position_mastery: 1,



Recommend Ban

In [119]:
from core.draft import recommend_pick

# Recommend a Pick
best, score, explanation, flag, all_results = recommend_pick(
    G_master, 
    "t2",
    "Jungler", 
    draft_state,
    global_data
)

print(f"Recommended: {best} ({score})")
for reason in explanation:
    output = f"- {reason}:"
    for hero in explanation[reason]:
        output += f" {hero},"
    print(output)
if flag:
    print(f"WARNING: {flag}")
print("")


Recommended: Aurelio (11.5)
- synergy_possible: Dylan,
- position_tier: S,
- position_mastery: 1,



Pick Heros

In [118]:
from core.draft import pick_hero

#pick_hero(G_master, "Aurelio", "t1", draft_state)
pick_hero(G_master, "Kamaitachi", "t2", draft_state)

In [52]:
from core.draft import ban_hero

ban_hero("Tiger Boy", draft_state)

In [71]:
from core.draft import see_current_draft

see_current_draft("t1", draft_state)

{'Peiniang Zhu': {'Support'},
 'Frank': {'Top'},
 'Zealot': {'Jungler'},
 'Gang': {'Bot'},
 'Aurelio': {'Mid'}}

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

from core.data import read_hero_info, read_role_itemisation


@tool
def get_hero_info(hero_name: str) -> str:
    """Get markdown notes on a specific hero by name.

    Args:
        hero_name (str): the name of the hero you want to look up
    """
    return read_hero_info(hero_name)


@tool
def get_role_itemisation(role: str) -> str:
    """Get markdown itemisation notes for a role or item archetype.

    Args:
        role (str): the role or item archetype to look up
    """
    return read_role_itemisation(role)


agent = create_agent(
    model="ollama:qwen3.5",
    tools=[get_hero_info, get_role_itemisation],
    system_prompt=(
        "You are a MOBA team coach. Questions given to you will be about "
        "strategy in drafts and hero playstyles."
    ),
)

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool

from core.data import read_hero_info, read_role_itemisation, read_glossary_definition


@tool
def get_hero_info(hero_name: str) -> str:
    """Get markdown notes on a specific hero by name.

    Args:
        hero_name (str): the name of the hero you want to look up

    Returns:
        str: Information regarding the hero.
    """
    return read_hero_info(hero_name)


@tool
def get_role_itemisation(role: str) -> str:
    """Get markdown itemisation notes for a role or item archetype.

    Args:
        role (str): the role or item archetype to look up

    Returns:
        str: Information regarding the role itemisation.
    """
    return read_role_itemisation(role)

@tool
def lookup_glossary(term: str) -> str:
    """Get term definitions from the glossary.

    Args:
        term (str): The term to lookup

    Returns:
        str: The definition of the term.
    """
    return read_glossary_definition(term)


SYSTEM_PROMPT = "" \
    "You are the Esports Godfather Super Coach.  You are here to guide the player in their drafting and playing strategies in the game Esports Godfather." \
    "If you recieve a question that is not related to heroes, strategies, glossary terminology, or similar, tell the user that you're an Esports Coach, and have no time to learn about anything else, therefore do not know the answer. " \
    "Do not mentioned `Based on the info provided`, or `the text states`.  You must present and speak as if you are the coach.  The information provided to you should be done silently." \

agent = create_agent(
    model="ollama:qwen3.5",
    tools=[get_hero_info, get_role_itemisation, lookup_glossary],
    system_prompt=SYSTEM_PROMPT
)

PROMPT = "The enemy currently has Palulu and Niels.  What should I pick as my first pick?  Any lane"

stream = agent.stream_events(
    {"messages": [{"role": "user", "content": PROMPT}]},
    version="v3"
)
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    if kind == "tool_calls":
        print(f"\nTool call: {item.tool_name}({item.input})")
        print("\n")


I need some information about these heroes and our team roles for this draft strategy! I'll investigate what kind of threats they present, but let's also think strategically here:

Let me check Palulu first to understand his abilities...
Tool call: get_hero_info({'hero_name': 'Palulu'})


Now let me check Niels...
Tool call: get_hero_info({'hero_name': 'Niels'})



Tool call: get_role_itemisation({'role': 'Support'})


Alright Coach here! So we've got Palulu (a support mage who throws shields) and Niels on their team. Now let me think about what's the best strategy for our first pick...

**Enemy Team Analysis:**
- **Palulu**: Provides healing/shields but otherwise doesn't contribute much combat capability - standard mid-tier support
- **Niels**: AoE glass cannon mage with 0 armor (highly vulnerable). He needs shields to avoid self-stunning and dying.

For our first pick, we need to think about:
1. What roles do they have covered? They likely picked a Support role slot each.
2. We still